In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

backend = BasicSimulator()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 53.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 8.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=f2b4dc3f84a82a20c938bbba82e4bd9ad93aa18b3c378980b08029e12463986c
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# function to generate ONE quantum random bit
# returns either 0 or 1

def quantum_random_bit():

  qc = QuantumCircuit(1,1)
  qc.h(0)
  qc.measure(0,0)
  compiled = transpile(qc, backend)
  result = backend.run(compiled, shots=1).result()
  counts = result.get_counts()
  bit = list(counts.keys())[0]
  return int(bit)

In [3]:
# generate multiple quantum random bits

def generate_quantum_bits(n):

    bits = []
    for _ in range(n):

        bits.append(
            quantum_random_bit()
        )
    return bits

In [4]:
# number of qubits
n = 20

# alice secret bits
alice_bits = generate_quantum_bits(n)

# alice random bases
# 0 = Z basis
# 1 = X basis
alice_bases = generate_quantum_bits(n)

print("alice bits:")
print(alice_bits)

print("\nalice bases:")
print(alice_bases)

alice bits:
[0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1]

alice bases:
[0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0]


In [5]:
# function to encode qubits

def encode_qubit(bit, basis):

    qc = QuantumCircuit(1,1)

    # if bit is 1 apply X gate
    if bit == 1:
        qc.x(0)

    # if basis is X basis apply Hadamard gate
    if basis == 1:
        qc.h(0)

    return qc

# store all encoded qubits
alice_qubits = []

# encode each bit
for bit, basis in zip(alice_bits, alice_bases):

    alice_qubits.append(
        encode_qubit(bit, basis)
    )

In [6]:
# bob chooses random bases

bob_bases = generate_quantum_bits(n)

print("bob bases:")
print(bob_bases)

bob bases:
[0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1]


In [7]:
# Function for Bob to measure qubits

def measure_qubit(qc, basis):

    # if bob uses X basis
    # apply Hadamard before measuring
    if basis == 1:
        qc.h(0)

    qc.measure(0,0)
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()
    counts = result.get_counts()
    bit = list(counts.keys())[0]
    return int(bit)

# bob measurement results
bob_results = []

# measure each qubit
for qc, basis in zip(alice_qubits, bob_bases):

    # copy circuit so original is unchanged
    new_qc = qc.copy()

    result = measure_qubit(new_qc, basis)

    bob_results.append(result)

print("bob results:")
print(bob_results)

bob results:
[0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0]


In [8]:
# shared keys
shared_key_alice = []
shared_key_bob = []

# compare bases
for i in range(n):

    # keep bits only if bases match
    if alice_bases[i] == bob_bases[i]:

        shared_key_alice.append(
            alice_bits[i]
        )

        shared_key_bob.append(
            bob_results[i]
        )

print("shared key (alice):")
print(shared_key_alice)

print("\nshared key (bob):")
print(shared_key_bob)

shared key (alice):
[0, 1, 0, 0, 0, 1, 0, 1, 0, 0]

shared key (bob):
[0, 1, 0, 0, 0, 1, 0, 1, 0, 0]


In [10]:
# count errors
errors = 0

for a, b in zip(shared_key_alice, shared_key_bob):

    if a != b:
        errors += 1

print("number of errors:", errors)

# no attacker means no errors
if errors == 0:

    print("\nno attacker detected.")

else:

    print("\nerrors found.")

number of errors: 0

no attacker detected.
